# Concorde — exploration et démonstration locale

Ce notebook inventorie les projets du serveur local, peut lancer une analyse complète du projet choisi, valide le JSON généré et vérifie les exports. Aucun contenu du défi ni aucune sortie calculée ne doit être enregistré dans le notebook ou poussé vers un service externe.

**Prérequis :** démarrez le serveur avec `start-concorde.ps1`. Le pipeline complet peut prendre longtemps et utiliser le GPU. La cellule d’analyse reste désactivée jusqu’à ce que `RUN_FULL_PIPELINE` soit mis à `True`.

In [ ]:
from pathlib import Path
import json, os, sys, time
from urllib.parse import urlparse, quote
from urllib.request import Request, urlopen
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if not (ROOT / 'l2c_app').is_dir():
    raise RuntimeError("Ouvrir le notebook depuis le dépôt Concorde ou son dossier notebooks.")
sys.path.insert(0, str(ROOT))
from l2c_app.models import Information
BASE_URL = os.environ.get("CONCORDE_URL", "http://127.0.0.1:" + os.environ.get("CONCORDE_PORT", "8766")).rstrip("/")
address = urlparse(BASE_URL)
if address.scheme != 'http' or address.hostname not in ('127.0.0.1', 'localhost') or address.username or address.password or address.path or address.query or address.fragment:
    raise ValueError("CONCORDE_URL doit désigner uniquement un serveur HTTP loopback local.")
def api_bytes(path, method="GET", value=None):
    body = None if value is None else json.dumps(value).encode("utf-8")
    headers = {"Accept": "application/json"}
    if body is not None: headers.update({"Content-Type": "application/json", "X-Concorde-Request": "1"})
    request = Request(BASE_URL + path, data=body, headers=headers, method=method)
    with urlopen(request, timeout=60) as response: return response.read()
def api_json(path, method="GET", value=None):
    return json.loads(api_bytes(path, method, value).decode("utf-8"))
overview = api_json("/api/overview")
assert overview.get("application") == "concorde", "Le port appartient à une autre application."
overview['hardware']


## Inventaire local

Choisir le projet uniquement parmi les PDF présents dans le catalogue. Les pages sans texte natif peuvent être des dessins vectoriels; elles ne sont pas comptées comme pages vides.

In [ ]:
projects = overview['projects']
[(p['id'], p['name'], p['plan_pages'], p['atelier_pages'], p['pages']) for p in projects]

## Pipeline complet ou vérification d’un résultat existant

Choisir un `PROJECT_ID` puis activer `RUN_FULL_PIPELINE` pour lancer le profil complet. Pour contrôler un résultat déjà calculé, renseigner seulement `RUN_ID` (les identifiants disponibles sont listés ci-dessous). Le notebook télécharge les exports via l’API locale : le kernel n’a pas besoin de partager le dossier `L2C_STORE` du serveur. Les contrôles sont en mémoire, aucune sortie confidentielle n’est écrite dans le dépôt.

Une interruption du kernel n’annule pas le calcul serveur. Retrouver sa progression dans Analyse; ne pas démarrer une seconde analyse. Pour le jury, importer le projet inédit dans l’interface puis rafraîchir l’inventaire. Ne pas présenter un résultat du cache comme une exécution chronométrée sur ce projet.


In [ ]:
PROJECT_ID = ''  # identifiant exact affiché dans l’inventaire
RUN_FULL_PIPELINE = False
RUN_ID = ''  # résultat existant à vérifier, facultatif
MAX_WAIT_SECONDS = 6 * 60 * 60
print([(j['id'], j.get('project_name', j.get('project_id')), j['status']) for j in overview['jobs']])
if RUN_FULL_PIPELINE:
    if PROJECT_ID not in {p['id'] for p in overview['projects']}:
        raise ValueError("Choisir un PROJECT_ID présent dans l’inventaire local.")
    job = api_json(f"/api/projects/{quote(PROJECT_ID, safe='')}/analyse", method="POST",
                   value={"profile": "complete", "max_ocr_pages": 1000})
    RUN_ID = job['id']
    print("Run", RUN_ID, "démarré")
    deadline, last_print = time.monotonic() + MAX_WAIT_SECONDS, 0
    while True:
        current = api_json(f"/api/jobs/{quote(RUN_ID, safe='')}")
        if time.monotonic() - last_print >= 10 or current['status'] not in ('queued', 'running'):
            print(f"{current['current']}/{current['total']} pages — {current['status']}")
            last_print = time.monotonic()
        if current['status'] not in ('queued', 'running'): break
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Délai du notebook atteint; run {RUN_ID} toujours actif sur le serveur.")
        time.sleep(2)
    if current['status'] != 'completed':
        raise RuntimeError(current.get('message', 'Analyse incomplète'))
elif not RUN_ID:
    print("Aucun calcul lancé. Choisir RUN_ID pour vérifier un résultat ou activer RUN_FULL_PIPELINE.")


## Contrôle des trois exports et de la couverture

Ces contrôles vérifient la structure et la couverture déclarée, pas la justesse métier. L’audit `scripts/audit_deliverables.py` vérifie en complément les fichiers/pages contre le catalogue. Les sorties restent à effacer avant toute remise du notebook.


In [ ]:
if RUN_ID:
    ident = quote(RUN_ID, safe='')
    job = api_json(f"/api/jobs/{ident}")
    assert job['status'] == 'completed', "Le résultat n’est pas encore exportable."
    result = api_json(f"/api/runs/{ident}/results?limit=1")
    prefix = f"/api/runs/{ident}/download/"
    exported = api_json(prefix + 'informations.json')
    assert isinstance(exported, list), "informations.json doit être une liste."
    for value in exported: Information.model_validate(value)
    assert len({value['id'] for value in exported}) == len(exported), "Identifiants dupliqués."
    comparisons = api_json(prefix + 'comparaisons.json')
    assert isinstance(comparisons, dict) and isinstance(comparisons.get("results"), list), "comparaisons.json doit contenir results."
    assert len(comparisons["results"]) == result["total"], "Compte des comparaisons incohérent."
    pdf = api_bytes(prefix + 'rapport.pdf')
    import pymupdf as fitz
    with fitz.open(stream=pdf, filetype='pdf') as report:
        assert len(report) > 0, "Rapport vide."
        report_pages = len(report)
    stats = result['statistics']
    assert len(exported) == stats['annotations'], "Compte des annotations incohérent."
    print("JSON valides:", len(exported), "annotations;", len(comparisons["results"]), "comparaisons; PDF:", report_pages, "pages")
    print("Portée:", result['scope'], "— couverture:", stats)
    complete = result['scope'] == 'complete' and stats['pages_processed'] == stats['pages_total'] and stats.get('pages_error', 0) == 0 and stats.get('pages_skipped', 0) == 0
    print("Couverture déclarée complète sans erreur:", complete)
    print("Cette vérification ne mesure pas la précision ni le rappel des écarts.")


## Exploration des informations extraites

La distribution par source/famille et la disponibilité des champs permettent de repérer un défaut de couverture. Un champ renseigné peut encore être incorrect; ces comptes ne sont pas des mesures de précision.


In [ ]:
if RUN_ID:
    from collections import Counter
    print("Annotations par source et famille:", dict(Counter((r['source'], r['type_element']) for r in exported)))
    bars = [bar for r in exported for bar in r['armature']]
    print("Groupes d’armatures:", len(bars))
    for field in ('repere', 'diametre', 'quantite', 'espacement_mm', 'longueur_mm'):
        present = sum(bar.get(field) is not None for bar in bars)
        print(field, present, '/', len(bars), 'renseignés')
    print("Statuts proposés:", dict(Counter(r['status'] for r in comparisons['results'])))


## Interpréter et valider

- Une exécution complète prouve la couverture des pages, pas l’exactitude de toutes les annotations. Consulter les erreurs et méthodes de lecture page par page.
- Vérifier des coordonnées sur les rotations 0°, 90°, 180° et 270° : elles doivent désigner le centre de l’annotation en points PDF depuis le coin supérieur gauche.
- Séparer CER/WER OCR, exactitude d’extraction, qualité d’appariement et précision/rappel des écarts.
- Les paires et non-conformités doivent être adjudiquées par un ingénieur. Garder toutes les pages/révisions d’un même projet ou dessin dans le même groupe; ne jamais répartir des tuiles corrélées entre les splits.
- Les quatre projets fournis ont déjà été explorés; ne pas les présenter comme un holdout complètement vierge. Le projet d’évaluation du jury n’est pas disponible avant la démo.